<a href="https://colab.research.google.com/github/KaliNangia/Audio-Transcript-Datasets-from-Youtube/blob/main/Helper_Notebook_to_download_Audio_from_YT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# YouTube Audio Transcription and Analysis

In [ ]:
%%capture
!pip install yt-dlp -U
# !apt-get update
!apt-get install -y ffmpeg

import pandas as pd
import yt_dlp

## Audio Download and Processing

This section provides functions to generate YouTube links and video IDs, and to download the audio from a given YouTube video. Run the following cells to set up these utilities.

In [ ]:
# %%capture #Comment out if you dont want any outputs
def YT_Link_Generator(VIDEO_ID :str) -> str:
    """Generator Function to generate YT_Link from VIDEO_ID"""
    return "https://www.youtube.com/watch?v=" + VIDEO_ID

def YT_VIDEO_ID_generator(VIDEO_LINK:str) -> str:
    """Generator function to generate VIDEO_IDS from Link"""
    return VIDEO_LINK[-11:]

def Download_Audio(video_ID):
    import yt_dlp # Import yt_dlp inside the function for multiprocessing

    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'wav',
            'preferredquality': '192',
        }],
        'outtmpl': f'/content/KrishiDarshan/{video_ID}/audio.%(ext)s'
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([YT_Link_Generator(video_ID)])

In [ ]:
# 1. Open an interactive input box to paste your link or ID
user_input = input("Enter YouTube Link or Video ID: ").strip()

# 2. Verify input is not empty, extract the ID, and download
if user_input:
    video_id = YT_VIDEO_ID_generator(user_input)
    print(f"Starting download for Video ID: {video_id}")
    Download_Audio(video_id)
else:
    print("Error: No link or ID was entered.")

### Gemini API Setup

These cells consolidate and reorder the Gemini API client initialization and model selection for a cleaner workflow, placed directly after the audio download. Please run these cells.

After running these, you can ignore the original separate cells for `!pip install -q google-genai`, Gemini client initialization (`1hpTKGYMy-db`), and the model selection dropdown (`17e160b3`), as their functionality is now integrated here. The cell `2fbbeba2` is also no longer necessary.

In [ ]:
!pip install -q google-genai

import google.colab.widgets as widgets
from ipywidgets import Dropdown
from google import genai
from google.colab import userdata

# Initialize the unified Gemini client
client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

# Test the connection
try:
    response = client.models.generate_content(
        model='gemini-2.5-flash', # Using flash for a quick test connection
        contents='Connection successful.'
    )
    print(response.text)
except Exception as e:
    print(f"Gemini API test failed: {e}. Please check your API key and quotas.")

# Define the available Gemini models
gemini_models = [
    'gemini-3.5-flash',
    'gemini-1.5-flash',
    'gemini-1.5-pro',
    'gemini-2.5-flash'
]

# Create a dropdown widget
model_selector = Dropdown(
    options=gemini_models,
    value='gemini-3.5-flash', # Default selected model
    description='Select Gemini Model:'
)

# Display the dropdown
display(model_selector)

# Assign the selected model to a variable
MODEL_NAME = model_selector.value
print(f"Selected model: {MODEL_NAME}")

# Update the MODEL_NAME when the dropdown value changes
def on_model_change(change):
    global MODEL_NAME
    MODEL_NAME = change.new
    print(f"Selected model updated to: {MODEL_NAME}")

model_selector.observe(on_model_change, names='value')

print("\nGemini API client initialized and model selector displayed.")

After running the above setup cells, please proceed to cell `59zp1X0S4GW8` (the transcription and chat cell). Remember to manually update it to use the `MODEL_NAME` variable (e.g., change `model='gemini-3.5-flash'` to `model=MODEL_NAME`) as instructed earlier, and then re-run it.

If you still encounter a `RESOURCE_EXHAUSTED` error, you will need to wait for your API quota to reset.

## Transcription and Chat

This section handles the audio transcription and allows for interactive questioning about the audio content. Before running the next cell, ensure you have selected your desired Gemini model from the dropdown above.

**Important:** Please manually update the transcription cell (`59zp1X0S4GW8`) to use the `MODEL_NAME` variable. Specifically, change `model='gemini-3.5-flash'` to `model=MODEL_NAME` in both the `generate_content` call for transcription and the `client.chats.create` call for the chat session. After making these changes, you can run the cell.

Remember that if you encounter a `RESOURCE_EXHAUSTED` error, you might need to wait for your API quota to reset.

In [ ]:
import time

# 1. Path to your downloaded audio file
file_path = f'/content/KrishiDarshan/{video_id}/audio.wav'

print("Uploading audio file to Gemini...")
audio_file = client.files.upload(file=file_path)

# Wait briefly for the file API to finish processing the state
print("Waiting for file processing...")
time.sleep(3)

print("\n--- Generating Transcript ---")
# 2. Generate the initial transcription
transcription_response = client.models.generate_content(
    model=MODEL_NAME,
    contents=[audio_file, "You are an expert, high-precision audio transcriber specializing in Indian agricultural media, specifically regional broadcasts like Krishi Darshan. Your task is to process the provided audio file and generate a highly detailed, chronological, verbatim transcript in hindi. Adhere strictly to the following execution rules: 1. SPEAKER IDENTIFICATION: Differentiate between speakers whenever the voice changes. If names are mentioned, use them (e.g., [Dr. Sharma]). If names are not explicitly mentioned, use logical identifiers like [Host], [Expert], or [Farmer]. 2. TIMESTAMPING RULES: Insert a precise timestamp at the beginning of EVERY speaker turn. If a single speaker talks continuously for more than 45 seconds, insert a new timestamp at the nearest natural sentence boundary. Use the exact format: [HH:MM:SS] (e.g., [00:04:12]). 3. VERBATIM FIDELITY: Transcribe the words exactly as they are spoken. Do not summarize, clean up grammar, or omit technical jargon, crop varieties, chemical names, or regional farming terms. If a technical word, brand name, or number is ambiguous or poorly pronounced, transcribe it to the best of your ability and append a [?] next to it (e.g., [Azotobacter?]). 4. OUTPUT FORMATTING: Present the final output as a clean, line-by-line log. Do not add markdown headers, introductory text, or conversational explanations. Ensure each entry follows this exact template: [TIMESTAMP] [SPEAKER]: Spoken text goes here. Example Output Line: [00:01:15] [Host]: Namaskar kisan bhaiyon, aaj hum baat karenge kharif ki fasal ke baare mein. [00:01:30] [Expert]: Kisan bhaiyon, is samay naye beejon ka chayan karna bahut mahatvapurna hai.."]
)
print(transcription_response.text)

print("\n--- Starting Conversation Mode ---")
# 3. Create an ongoing chat session with the audio context
chat = client.chats.create(model=MODEL_NAME)

# FIXED: Changed 'contents=' to 'message=' to align with the new SDK
chat.send_message(message=[audio_file, "Analyze this audio file. I will now ask you questions about its contents."])

print("Chat initialized! You can now ask questions about the audio below. (Type 'exit' to stop)")

# 4. Interactive loop to converse with the audio
while True:
    user_input = input("\nYour Question: ")
    if user_input.lower() == 'exit':
        print("Ending chat session.")
        break

    if user_input.strip() == '':
        continue

    response = chat.send_message(user_input)
    print(f"\nGemini: {response.text}")

## Save Transcript to Google Drive

This section will save the generated transcript to your Google Drive. Please ensure your Google Drive is mounted and that the `video_id` variable (from the audio download step) is correctly set. The transcript will be saved in a specific folder structure within your Drive.

In [ ]:
# The model names in the existing cell need to be updated to use the MODEL_NAME variable.
# Please manually update the model name in the cell below to `MODEL_NAME`.
# For example, change `model='gemini-3.5-flash'` to `model=MODEL_NAME`
# and `client.chats.create(model='gemini-3.5-flash')` to `client.chats.create(model=MODEL_NAME)`

In [ ]:
import os
from google.colab import drive

# 1. Mount Google Drive if it isn't already mounted
try:
    drive.mount('/content/drive')
except Exception:
    pass

# 2. Your precise destination base directory path
base_path = "/content/drive/MyDrive/AnnamAI Tasks/Golden Database for Transcript"

# 3. Automatically inherits the video_id variable from the previous download cell
if 'video_id' not in locals() and 'video_id' not in globals():
    raise NameError("Variable 'video_id' not found. Please make sure to run your interactive download cell first!")

# 4. Dynamically append the video ID folder to your base path
target_folder = os.path.join(base_path, video_id)

# 5. Automatically create the directory if it doesn't exist yet
os.makedirs(target_folder, exist_ok=True)

# 6. Define the full text file destination
transcript_path = os.path.join(target_folder, "transcript.txt")

# 7. Write a new file or completely overwrite/update the old file with the latest text
with open(transcript_path, "w", encoding="utf-8") as f:
    f.write(transcription_response.text)

print(f"✨ Success! transcript.txt file added: {transcript_path}")